# LeetCode 1180 — Count Substrings with Only One Distinct Letter bằng Python

**Link đề:** https://leetcode.com/problems/count-substrings-with-only-one-distinct-letter/
**Độ khó:** Easy · **Chủ đề:** Math, String

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def countLetters(self, s: str) -> int:
        ...
```

## Mục tiêu học
1. Phân biệt **substring** (chuỗi con liên tiếp) với subsequence; đếm số substring của chuỗi dài `L`: `L(L+1)/2`.
2. Brute force: duyệt mọi substring và kiểm tra bằng `set` — thấy vì sao `O(n³)`.
3. Kỹ thuật **chia chuỗi thành các "run"** (đoạn ký tự giống nhau liên tiếp) — run-length.
4. Công thức tổng `1 + 2 + ... + L` và mẹo **đếm một lượt**: mỗi vị trí đóng góp đúng **độ dài run hiện tại**.
5. Dùng `itertools.groupby` để nhóm run gọn gàng.
6. Trace, chứng minh, test đối chiếu, lỗi hay gặp, biến thể, bài tập.

## 1. Đề bài và ví dụ

Cho chuỗi `s`. Trả về **số substring** chỉ chứa **một loại ký tự** (mọi ký tự trong substring đều giống nhau).

| `s` | Output | Giải thích |
|---|---:|---|
| `"aaaba"` | `8` | `"aaa"` ×1, `"aa"` ×2, `"a"` ×4, `"b"` ×1 → 1+2+4+1 = 8 |
| `"aaaaaaaaaa"` | `55` | `10·11/2 = 55` |

**Constraints (theo hiểu biết):**
- `1 <= s.length <= 1000`
- `s[i]` là chữ cái thường tiếng Anh.

Lưu ý: substring được đếm **theo vị trí**: `"a"` ở index 0 và `"a"` ở index 1 là hai substring khác nhau.

In [1]:
s = "aaaba"
found = []
for i in range(len(s)):
    for j in range(i + 1, len(s) + 1):
        sub = s[i:j]
        if len(set(sub)) == 1:
            found.append((i, j, sub))
for item in found:
    print(item)
print("tổng =", len(found))

(0, 1, 'a')
(0, 2, 'aa')
(0, 3, 'aaa')
(1, 2, 'a')
(1, 3, 'aa')
(2, 3, 'a')
(3, 4, 'b')
(4, 5, 'a')
tổng = 8


## 2. Nền tảng Python cần biết

### 2.1. Substring và slicing `s[i:j]`

Substring là đoạn **liên tiếp** `s[i:j]` với `0 <= i < j <= n`. Một chuỗi dài `n` có `n(n+1)/2` substring (không rỗng): chọn `i` và `j`.

In [2]:
s = "abc"
subs = [s[i:j] for i in range(len(s)) for j in range(i + 1, len(s) + 1)]
print(subs, len(subs), "= 3*4/2 =", 3 * 4 // 2)

['a', 'ab', 'abc', 'b', 'bc', 'c'] 6 = 3*4/2 = 6


### 2.2. `set` — tập hợp không trùng

`set("aab")` → `{'a', 'b'}`. Substring chỉ có một loại ký tự ⇔ `len(set(sub)) == 1`.

In [3]:
print(set("aab"), len(set("aab")))
print(set("aaa"), len(set("aaa")))

{'b', 'a'} 2
{'a'} 1


### 2.3. Tổng `1 + 2 + ... + L = L(L+1)/2`

Mẹo Gauss: viết tổng xuôi và ngược rồi cộng cặp: mỗi cặp bằng `L + 1`, có `L` cặp → `2·Tổng = L(L+1)`.

Dùng `//` để giữ kết quả là số nguyên (vì `L(L+1)` luôn chẵn).

In [4]:
for L in [1, 2, 3, 4, 10]:
    print(L, sum(range(1, L + 1)), L * (L + 1) // 2)

1 1 1
2 3 3
3 6 6
4 10 10
10 55 55


### 2.4. `itertools.groupby` — nhóm các phần tử **liên tiếp** giống nhau

`groupby("aaaba")` sinh các cặp `(key, nhóm)`: `('a', 'aaa')`, `('b', 'b')`, `('a', 'a')`. Chú ý nó chỉ gộp phần tử **kề nhau**, nên hai nhóm `'a'` tách riêng — đúng thứ ta cần.

In [5]:
from itertools import groupby

for key, group in groupby("aaaba"):
    g = list(group)
    print(key, g, "độ dài", len(g))

a ['a', 'a', 'a'] độ dài 3
b ['b'] độ dài 1
a ['a'] độ dài 1


## 3. Cách 1 — Brute force: mọi substring + `set`

In [6]:
def count_letters_brute(s: str) -> int:
    count = 0
    n = len(s)
    for i in range(n):
        for j in range(i + 1, n + 1):
            if len(set(s[i:j])) == 1:
                count += 1
    return count

print(count_letters_brute("aaaba"), count_letters_brute("aaaaaaaaaa"))

8 55


### Nút thắt
- `n(n+1)/2` substring, mỗi lần cắt + tạo set tốn `O(n)` → **`O(n³)`**. Với `n = 1000`: ~`1.7·10^8` thao tác → chậm.
- Lãng phí: nếu `s[i:j]` đã có 2 loại ký tự thì mọi `s[i:j']` dài hơn cũng vậy — **không cần xét tiếp**.

## 4. Cách 2 — Mở rộng từ mỗi điểm bắt đầu (O(n²))

Từ mỗi `i`, kéo `j` sang phải **chừng nào** `s[j] == s[i]`; mỗi bước được một substring hợp lệ. Gặp ký tự khác thì dừng.

In [7]:
def count_letters_expand(s: str) -> int:
    count = 0
    n = len(s)
    for i in range(n):
        j = i
        while j < n and s[j] == s[i]:
            count += 1          # s[i:j+1] hợp lệ
            j += 1
    return count

print(count_letters_expand("aaaba"), count_letters_expand("aaaaaaaaaa"))

8 55


Vẫn `O(n²)` khi chuỗi toàn một ký tự (`"aaaa..."`). Ta cần quan sát cấu trúc sâu hơn.

## 5. Insight — Chuỗi tách thành các "run"

Một substring chỉ có một loại ký tự **phải nằm gọn trong một run** (đoạn liên tiếp các ký tự giống nhau).

```text
"aaaba"  →  run "aaa" (L=3) | run "b" (L=1) | run "a" (L=1)
```

Một run dài `L` có `L(L+1)/2` substring, và **mọi** substring của nó đều hợp lệ. Vậy:

```text
đáp án = Σ L(L+1)/2 trên mọi run
"aaaba" → 3·4/2 + 1·2/2 + 1·2/2 = 6 + 1 + 1 = 8 ✔
```

In [8]:
s = "aaaba"
runs = [len(list(g)) for _, g in groupby(s)]
print("độ dài các run:", runs)
print("đóng góp:", [L * (L + 1) // 2 for L in runs], "→ tổng", sum(L * (L + 1) // 2 for L in runs))

độ dài các run: [3, 1, 1]
đóng góp: [6, 1, 1] → tổng 8


## 6. Cách 3 — Run-length tự viết + công thức

**Mổ xẻ:**
1. `run = 1` (độ dài run hiện tại, bắt đầu với `s[0]`).
2. Với `i` từ 1: nếu `s[i] == s[i-1]` → `run += 1`; ngược lại run cũ **kết thúc**: cộng `run(run+1)/2`, đặt `run = 1`.
3. Sau vòng lặp, **đừng quên** cộng run cuối cùng.

In [9]:
def count_letters_runs(s: str) -> int:
    total = 0
    run = 1
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            run += 1
        else:
            total += run * (run + 1) // 2
            run = 1
    total += run * (run + 1) // 2      # run cuối cùng
    return total

print(count_letters_runs("aaaba"), count_letters_runs("aaaaaaaaaa"), count_letters_runs("a"))

8 55 1


### Vì sao một run dài `L` có đúng `L(L+1)/2` substring?

Đếm theo **độ dài** substring trong run `"aaaa"` (`L = 4`):

| độ dài | các substring (theo vị trí bắt đầu) | số lượng |
|---:|---|---:|
| 1 | `a`@0, `a`@1, `a`@2, `a`@3 | 4 |
| 2 | `aa`@0, `aa`@1, `aa`@2 | 3 |
| 3 | `aaa`@0, `aaa`@1 | 2 |
| 4 | `aaaa`@0 | 1 |

Tổng `4 + 3 + 2 + 1 = 10 = 4·5/2`. Tổng quát: độ dài `d` có `L - d + 1` vị trí bắt đầu → `Σ (L - d + 1) = L + (L-1) + ... + 1`.

**Câu hỏi tự kiểm tra:** hai run cùng ký tự nhưng **không kề nhau** (như hai chữ `a` trong `"aaaba"`) có được gộp lại không? → Không, vì substring nối chúng phải đi qua chữ `b`.

In [10]:
def substrings_by_length(L: int) -> None:
    run = "a" * L
    total = 0
    for d in range(1, L + 1):
        starts = [i for i in range(L - d + 1)]
        total += len(starts)
        print(f"độ dài {d}: bắt đầu tại {starts} -> {len(starts)} substring")
    print(f"tổng = {total} = L(L+1)/2 = {L * (L + 1) // 2}")

substrings_by_length(4)

độ dài 1: bắt đầu tại [0, 1, 2, 3] -> 4 substring
độ dài 2: bắt đầu tại [0, 1, 2] -> 3 substring
độ dài 3: bắt đầu tại [0, 1] -> 2 substring
độ dài 4: bắt đầu tại [0] -> 1 substring
tổng = 10 = L(L+1)/2 = 10


## 7. Cách 4 — Lời giải chính: cộng dồn độ dài run tại mỗi vị trí

Nhìn theo **điểm kết thúc**: số substring hợp lệ **kết thúc tại vị trí `i`** chính là độ dài run tính tới `i` (`cur`). Ví dụ trong `"aaa"`, tại index 2 có `"a"`, `"aa"`, `"aaa"` → 3 = `cur`.

Vậy chỉ cần một lượt: cập nhật `cur` rồi `total += cur`. Không cần công thức, không cần xử lý run cuối riêng.

```text
1 + 2 + ... + L  =  L(L+1)/2   (cộng dần chính là công thức)
```

In [11]:
def count_letters(s: str) -> int:
    total = 0
    cur = 0
    for i, ch in enumerate(s):
        if i > 0 and ch == s[i - 1]:
            cur += 1
        else:
            cur = 1
        total += cur          # số substring hợp lệ kết thúc tại i
    return total

print(count_letters("aaaba"), count_letters("aaaaaaaaaa"))

8 55


`i > 0 and ch == s[i - 1]`: điều kiện `i > 0` đứng trước để khi `i = 0` không so với `s[-1]` (ký tự **cuối** chuỗi trong Python — chỉ số âm!). Python không báo lỗi khi dùng chỉ số âm, nên đây là loại bug "âm thầm": ở bản này nhờ `cur` khởi tạo bằng 0 nên tình cờ vô hại, nhưng chỉ cần khởi tạo khác (ví dụ `cur = 1` cho "run đầu") là `"aba"` sẽ bị coi như `s[0]` nối tiếp `s[-1]`. Hãy luôn viết điều kiện biên rõ ràng.

In [12]:
s = "aba"
print("s[-1] =", s[-1])   # chỉ số âm không lỗi mà lấy từ cuối → dễ gây bug âm thầm

s[-1] = a


## 8. TRACE từng bước với `s = "aaaba"`

| i | ch | cùng ký tự trước? | cur | total |
|---:|:---:|---|---:|---:|
| 0 | a | (đầu chuỗi) | 1 | 1 |
| 1 | a | có | 2 | 3 |
| 2 | a | có | 3 | 6 |
| 3 | b | không | 1 | 7 |
| 4 | a | không | 1 | 8 |

In [13]:
def trace_count_letters(s: str) -> int:
    total = cur = 0
    for i, ch in enumerate(s):
        same = i > 0 and ch == s[i - 1]
        cur = cur + 1 if same else 1
        total += cur
        ends = [s[i - L + 1:i + 1] for L in range(1, cur + 1)]
        print(f"i={i} ch={ch!r} same={same!s:<5} cur={cur} total={total:>2} | kết thúc tại i: {ends}")
    return total

trace_count_letters("aaaba")
print()
trace_count_letters("zzyzz")

i=0 ch='a' same=False cur=1 total= 1 | kết thúc tại i: ['a']
i=1 ch='a' same=True  cur=2 total= 3 | kết thúc tại i: ['a', 'aa']
i=2 ch='a' same=True  cur=3 total= 6 | kết thúc tại i: ['a', 'aa', 'aaa']
i=3 ch='b' same=False cur=1 total= 7 | kết thúc tại i: ['b']
i=4 ch='a' same=False cur=1 total= 8 | kết thúc tại i: ['a']

i=0 ch='z' same=False cur=1 total= 1 | kết thúc tại i: ['z']
i=1 ch='z' same=True  cur=2 total= 3 | kết thúc tại i: ['z', 'zz']
i=2 ch='y' same=False cur=1 total= 4 | kết thúc tại i: ['y']
i=3 ch='z' same=False cur=1 total= 5 | kết thúc tại i: ['z']
i=4 ch='z' same=True  cur=2 total= 7 | kết thúc tại i: ['z', 'zz']


7

Cột cuối liệt kê đúng các substring hợp lệ **kết thúc tại `i`** — có đúng `cur` cái. Đây là lý do `total += cur` đếm đủ và không trùng.

`cur + 1 if same else 1` là **biểu thức điều kiện** (ternary): `A if điều_kiện else B`.

## 9. Cách 5 — `groupby` một dòng

In [14]:
def count_letters_groupby(s: str) -> int:
    return sum(L * (L + 1) // 2 for L in (len(list(g)) for _, g in groupby(s)))

print(count_letters_groupby("aaaba"), count_letters_groupby("aaaaaaaaaa"))

8 55


`for _, g in groupby(s)`: dùng `_` cho `key` vì ta không cần biết ký tự là gì, chỉ cần độ dài nhóm.

## 10. Code nộp LeetCode

In [15]:
class Solution:
    def countLetters(self, s: str) -> int:
        total = 0
        cur = 0
        for i in range(len(s)):
            if i > 0 and s[i] == s[i - 1]:
                cur += 1
            else:
                cur = 1
            total += cur
        return total

sol = Solution()
print(sol.countLetters("aaaba"), sol.countLetters("aaaaaaaaaa"))

8 55


In [16]:
from itertools import groupby

class SolutionGroupby:
    def countLetters(self, s: str) -> int:
        total = 0
        for _, g in groupby(s):
            L = sum(1 for _ in g)
            total += L * (L + 1) // 2
        return total

print(SolutionGroupby().countLetters("aaaba"))

8


## 11. Vì sao đúng?

**Phân hoạch theo điểm kết thúc:** mỗi substring hợp lệ có **đúng một** điểm kết thúc `i`. Vậy đáp án = Σ (số substring hợp lệ kết thúc tại `i`).

**Bổ đề:** số substring hợp lệ kết thúc tại `i` = độ dài run chứa `s[i]` tính từ đầu run tới `i` (gọi là `cur_i`).
- Substring `s[j..i]` hợp lệ ⇔ `s[j] = ... = s[i]` ⇔ `j` nằm trong run hiện tại ⇔ `i - cur_i + 1 <= j <= i` → đúng `cur_i` lựa chọn.

**Invariant vòng lặp:** sau khi xử lý `i`, `cur = cur_i` và `total` = số substring hợp lệ kết thúc tại một vị trí `<= i`. Cập nhật `cur` đúng vì: nếu `s[i] == s[i-1]` thì run kéo dài thêm 1, ngược lại run mới bắt đầu với độ dài 1.

## 12. Độ phức tạp

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| Brute force + set | `O(n³)` | `O(n)` |
| Mở rộng từ mỗi i | `O(n²)` xấu nhất | `O(1)` |
| Run + công thức | `O(n)` | `O(1)` |
| Cộng dồn `cur` | `O(n)` | `O(1)` |
| `groupby` | `O(n)` | `O(1)` (bản `sum(1 for _ in g)`) / `O(L)` (bản `list(g)`) |

Đáp án lớn nhất (chuỗi toàn một ký tự, `n = 1000`) là `1000·1001/2 = 500500` — vừa số nguyên 32-bit.

## 13. Bộ test: mẫu, biên, đối chiếu ngẫu nhiên

In [17]:
cases = [
    ("aaaba", 8),
    ("aaaaaaaaaa", 55),
    ("a", 1),
    ("ab", 2),
    ("abc", 3),
    ("aab", 4),
    ("abb", 4),
    ("aba", 3),          # hai chữ 'a' không liền nhau
    ("zzyzz", 7),
    ("a" * 1000, 500500),
]
solvers = [count_letters_brute, count_letters_expand, count_letters_runs, count_letters,
           count_letters_groupby, Solution().countLetters, SolutionGroupby().countLetters]
for s, expected in cases:
    for f in solvers:
        if f is count_letters_brute and len(s) > 50:
            continue                      # brute O(n^3) quá chậm với n = 1000
        assert f(s) == expected, (f, s)
    print(f"{s[:12]!r:<16} -> {expected} PASS")

'aaaba'          -> 8 PASS
'aaaaaaaaaa'     -> 55 PASS
'a'              -> 1 PASS
'ab'             -> 2 PASS
'abc'            -> 3 PASS
'aab'            -> 4 PASS
'abb'            -> 4 PASS
'aba'            -> 3 PASS
'zzyzz'          -> 7 PASS
'aaaaaaaaaaaa'   -> 500500 PASS


In [18]:
import random
random.seed(1180)
for _ in range(1000):
    s = "".join(random.choice("ab") for _ in range(random.randint(1, 20)))
    expected = count_letters_brute(s)
    for f in solvers[1:]:
        assert f(s) == expected, (f, s)
print("1000 chuỗi ngẫu nhiên khớp brute force. PASS")

1000 chuỗi ngẫu nhiên khớp brute force. PASS


## 14. Lỗi người mới hay gặp

### Lỗi 1: Quên cộng run cuối cùng (bản run + công thức)
```python
for i in range(1, n):
    if s[i] == s[i-1]: run += 1
    else: total += run*(run+1)//2; run = 1
return total                 # thiếu run cuối → "aaaa" ra 0
```

### Lỗi 2: Đếm số **loại** ký tự hoặc số run thay vì số substring.

### Lỗi 3: Gặp ký tự mới thì đặt `cur = 0` thay vì `cur = 1`
Ký tự mới tự nó đã là một substring hợp lệ (độ dài 1), nên run mới có độ dài **1**.

### Lỗi 4: Dùng `/` thay vì `//` → kết quả là số thực `6.0`.

In [19]:
def wrong_forget_last(s):
    total, run = 0, 1
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            run += 1
        else:
            total += run * (run + 1) // 2
            run = 1
    return total

def wrong_reset_zero(s):
    total = cur = 0
    for i in range(len(s)):
        cur = cur + 1 if i > 0 and s[i] == s[i - 1] else 0   # sai: run mới phải dài 1
        total += cur
    return total

def wrong_float(s):
    return sum(len(list(g)) * (len(list(g)) + 1) / 2 for _, g in groupby(s))

print("aaaa: đúng", count_letters("aaaa"), "| quên run cuối:", wrong_forget_last("aaaa"))
print("aaaba: đúng", count_letters("aaaba"), "| đặt lại cur = 0:", wrong_reset_zero("aaaba"))
print("aaa : đúng", count_letters("aaa"), "| dùng / và gọi list(g) hai lần:", wrong_float("aaa"))

aaaa: đúng 10 | quên run cuối: 0
aaaba: đúng 8 | đặt lại cur = 0: 3
aaa : đúng 6 | dùng / và gọi list(g) hai lần: 1.5


Lỗi cuối có **hai** vấn đề: `/` cho số thực, và `g` là **iterator** — gọi `list(g)` lần thứ hai thì nó đã cạn, trả `[]`, nên độ dài lần hai thành 0: `3 * (0 + 1) / 2 = 1.5` thay vì `6`. Luôn lưu `L = len(list(g))` **một lần**.

## 15. Mẹo nhận dạng pattern

- "Đếm substring thỏa điều kiện chỉ phụ thuộc vào **đoạn liên tiếp giống nhau**" → chia **run**, dùng `L(L+1)/2` hoặc cộng dồn độ dài run.
- "Đếm theo **điểm kết thúc**" là kỹ thuật chung giúp tránh đếm trùng: đáp án = Σ (số đoạn hợp lệ kết thúc tại `i`).
- `itertools.groupby` = công cụ run-length nhanh trong Python.

Bài cùng họ: 1513 (Number of Substrings With Only 1s), 1759 (Count Number of Homogenous Substrings), 413 (Arithmetic Slices), 2348 (Number of Zero-Filled Subarrays).

## 16. Biến thể

### 16.1. Đếm subarray toàn số 0 (2348) — cùng ý tưởng trên mảng số

In [20]:
def zero_filled_subarrays(nums) -> int:
    total = cur = 0
    for x in nums:
        cur = cur + 1 if x == 0 else 0
        total += cur
    return total

print(zero_filled_subarrays([1, 3, 0, 0, 2, 0, 0, 4]))   # 6

6


### 16.2. Run-length encoding: `"aaaba"` → `"a3b1a1"`

In [21]:
def rle(s: str) -> str:
    return "".join(f"{ch}{len(list(g))}" for ch, g in groupby(s))

print(rle("aaaba"), rle("zzyzz"))

a3b1a1 z2y1z2


### 16.3. Có modulo (1759 Count Homogenous Substrings)
Giống hệt bài này nhưng `n` tới `10^5` và trả `% (10^9 + 7)` — chỉ cần thêm modulo vào `total`.

### 16.4. So sánh các cách

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| Brute + set | thử mọi substring | `O(n³)` | `O(n)` | kiểm chứng |
| Mở rộng | dừng khi khác ký tự | `O(n²)` | `O(1)` | cải tiến tự nhiên |
| Run + công thức | `Σ L(L+1)/2` | `O(n)` | `O(1)` | nhớ run cuối |
| Cộng dồn `cur` | đếm theo điểm kết thúc | `O(n)` | `O(1)` | **nộp bài** |
| `groupby` | run-length có sẵn | `O(n)` | `O(1)` | ngắn gọn |

## 17. Bài tập vận dụng (TODO)

### Bài 1 — Tính tay
`s = "aabbbc"`: liệt kê độ dài các run và tính đáp án.

### Bài 2 — Trả về danh sách độ dài run (không dùng groupby)

In [22]:
def run_lengths_exercise(s: str) -> list:
    # TODO: duyệt, so với ký tự trước; hết run thì append
    pass

# assert run_lengths_exercise("aaaba") == [3, 1, 1]
# assert run_lengths_exercise("z") == [1]

### Bài 3 — Hoàn thiện lời giải cộng dồn

In [23]:
def count_letters_exercise(s: str) -> int:
    # TODO: cur = độ dài run tới i; total += cur
    pass

# assert count_letters_exercise("aaaba") == 8
# assert count_letters_exercise("aaaaaaaaaa") == 55

### Bài 4 — Number of Substrings With Only 1s (1513)
Chuỗi nhị phân, đếm substring toàn `'1'`, trả `% (10**9 + 7)`.

In [24]:
def num_sub_ones_exercise(s: str) -> int:
    # TODO
    pass

# assert num_sub_ones_exercise("0110111") == 9
# assert num_sub_ones_exercise("000") == 0

### Bài 5 — Substring "đồng nhất" dài nhất
Trả về độ dài run dài nhất.

In [25]:
def longest_run_exercise(s: str) -> int:
    # TODO
    pass

# assert longest_run_exercise("aaaba") == 3
# assert longest_run_exercise("abc") == 1

### Bài 6 — Mini challenge: đếm theo từng ký tự
Trả về dict `{ký tự: số substring đồng nhất gồm ký tự đó}`.

In [26]:
def count_by_letter_exercise(s: str) -> dict:
    # TODO: cộng L(L+1)/2 vào đúng key của run
    pass

# assert count_by_letter_exercise("aaaba") == {"a": 7, "b": 1}

## 18. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | tách run | 2, 3, 1 | `3 + 6 + 1` |
| 2 | biến `run` | so `s[i]` với `s[i-1]` | nhớ `append(run)` sau vòng lặp |
| 3 | xem mục 7 | `cur = cur + 1 if same else 1` | `total += cur` |
| 4 | chỉ run ký tự `'1'` | `cur = cur + 1 if ch == '1' else 0` | `total = (total + cur) % MOD` |
| 5 | max của `cur` | `best = max(best, cur)` | khởi tạo `best = 0` |
| 6 | groupby cho cả key | `ch, g` | `d[ch] = d.get(ch, 0) + L*(L+1)//2` |

## 19. Đáp án tham khảo

**Bài 1:** run `"aa"` (2), `"bbb"` (3), `"c"` (1) → `3 + 6 + 1 = ` **10**.

In [27]:
def run_lengths_answer(s: str) -> list:
    res = []
    run = 1
    for i in range(1, len(s)):
        if s[i] == s[i - 1]:
            run += 1
        else:
            res.append(run)
            run = 1
    res.append(run)
    return res

def count_letters_answer(s: str) -> int:
    total = cur = 0
    for i in range(len(s)):
        cur = cur + 1 if i > 0 and s[i] == s[i - 1] else 1
        total += cur
    return total

def num_sub_ones_answer(s: str) -> int:
    MOD = 10**9 + 7
    total = cur = 0
    for ch in s:
        cur = cur + 1 if ch == "1" else 0
        total = (total + cur) % MOD
    return total

def longest_run_answer(s: str) -> int:
    best = cur = 0
    for i in range(len(s)):
        cur = cur + 1 if i > 0 and s[i] == s[i - 1] else 1
        best = max(best, cur)
    return best

def count_by_letter_answer(s: str) -> dict:
    d = {}
    for ch, g in groupby(s):
        L = len(list(g))
        d[ch] = d.get(ch, 0) + L * (L + 1) // 2
    return d

assert Solution().countLetters("aabbbc") == 10
assert run_lengths_answer("aaaba") == [3, 1, 1]
assert run_lengths_answer("z") == [1]
assert count_letters_answer("aaaba") == 8
assert count_letters_answer("aaaaaaaaaa") == 55
assert num_sub_ones_answer("0110111") == 9
assert num_sub_ones_answer("000") == 0
assert longest_run_answer("aaaba") == 3
assert longest_run_answer("abc") == 1
assert count_by_letter_answer("aaaba") == {"a": 7, "b": 1}
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 20. Template nhớ nhanh + checklist submit

```python
class Solution:
    def countLetters(self, s: str) -> int:
        total = cur = 0
        for i in range(len(s)):
            cur = cur + 1 if i > 0 and s[i] == s[i - 1] else 1
            total += cur
        return total
```

**Checklist:**
- [ ] Đếm substring **theo vị trí** (các `"a"` ở vị trí khác nhau là khác nhau)?
- [ ] Không so `s[0]` với `s[-1]`?
- [ ] Nếu dùng công thức theo run: đã cộng **run cuối**?
- [ ] Dùng `//` (số nguyên)?
- [ ] Test `"a"`, `"aba"`, chuỗi toàn một ký tự dài 1000?